In [1]:
import numpy as np
import os
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.datasets import Planetoid

from sklearn.metrics import accuracy_score

In [2]:
dataset = Planetoid(root='data/Planetoid', name='Cora')
data = dataset[0]
print('data')

print('torch.min(data.y) = ' + str(torch.min(data.y)))
print('torch.max(data.y) = ' + str(torch.max(data.y)))
### you can convert it into probabilities?
nlabel = torch.max(data.y) - torch.min(data.y) + 1
eps = 1.0e-6
yprob = eps + torch.zeros((data.y.shape[0], nlabel))
for idx in range(yprob.shape[0]):
    yprob[idx, data.y[idx]] = 1.0 - (nlabel - 1) * eps
print(data.x.shape)
print(data.y.shape)
print(yprob[0, :])

data
torch.min(data.y) = tensor(0)
torch.max(data.y) = tensor(6)
torch.Size([2708, 1433])
torch.Size([2708])
tensor([1.0000e-06, 1.0000e-06, 1.0000e-06, 9.9999e-01, 1.0000e-06, 1.0000e-06,
        1.0000e-06])


In [3]:
dataset = Planetoid(root='data/Planetoid', name='Cora')
data = dataset[0]
print('data')

print('torch.min(data.y) = ' + str(torch.min(data.y)))
print('torch.max(data.y) = ' + str(torch.max(data.y)))
### you can convert it into probabilities?
nlabel = torch.max(data.y) - torch.min(data.y) + 1
eps = 1.0e-6
yprob = eps + torch.zeros((data.y.shape[0], nlabel))
for idx in range(yprob.shape[0]):
    yprob[idx, data.y[idx]] = 1.0 - (nlabel - 1) * eps
print(data.x.shape)
ylabel = 0 + data.y
data.y = yprob


data
torch.min(data.y) = tensor(0)
torch.max(data.y) = tensor(6)
torch.Size([2708, 1433])


In [4]:
class SimpleGCN(torch.nn.Module):
    def __init__(self, num_nodes, emb_dim):
        super(SimpleGCN, self).__init__()
        self.emb = nn.Embedding(num_nodes, emb_dim)
        # GCNConv layer: maps input features (1) to output features (16)
        self.conv1 = GCNConv(in_channels=emb_dim, out_channels=16)
        # Another GCNConv layer: maps 16 features to 2 (e.g., for 2 classes in node classification)
        self.conv2 = GCNConv(in_channels=16, out_channels=7)

    def forward(self, data, idx_train):
        x0, edge_index, y = data.x, data.edge_index, data.y
        x = self.emb.weight
        print(x[0, :])
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        log_probs = torch.log_softmax(x[idx_train, :], dim=1)
        loss = torch.mean(-torch.sum(y[idx_train, :] * log_probs, dim=1))
        # loss = -log_probs[torch.arange(x.size(0)), y]
        return (log_probs, loss)

In [5]:
num_nodes, emb_dim = data.x.shape
emb_dim = 16
model = SimpleGCN(num_nodes, emb_dim)
opt = torch.optim.Adam(model.parameters(), lr=1e-2)

perm = np.random.permutation(num_nodes)
frac_train = 0.1
ntrain = np.int64(frac_train * num_nodes)
idx_train = perm[:ntrain]
idx_ttest = perm[ntrain:]
ltrain_true = ylabel[idx_train[:]]
lttest_true = ylabel[idx_ttest[:]]




nepoch = 60
model.emb.requires_grad_(True)
model.conv1.requires_grad_(True)
model.conv2.requires_grad_(True)
for kepoch in range(nepoch):
    log_prob, loss = model(data, idx_train)
    opt.zero_grad()
    loss.backward()
    opt.step()
    print('loss = ' + str(loss.item()))
log_prob_train, _ = model(data, idx_train)
log_prob_ttest, _ = model(data, idx_ttest)
log_prob_train = log_prob_train.detach().cpu().numpy()
log_prob_ttest = log_prob_ttest.detach().cpu().numpy()
ltrain_true = ltrain_true.detach().cpu().numpy()
ltrain_test = lttest_true.detach().cpu().numpy()
ltrain_pred = np.argmax(log_prob_train, axis=1)
lttest_pred = np.argmax(log_prob_ttest, axis=1)
acc_train = accuracy_score(ltrain_true, ltrain_pred)
acc_ttest = accuracy_score(lttest_true, lttest_pred)
print('acc_train = ' + str(acc_train))
print('acc_ttest = ' + str(acc_ttest))


tensor([ 1.2581,  0.7791,  0.6846, -0.4956, -0.3729,  0.1310,  2.7467, -0.1967,
        -0.6169,  0.1559,  0.1666,  0.8203, -1.3056, -0.5801,  0.9552,  0.3361],
       grad_fn=<SliceBackward0>)
loss = 2.0270774364471436
tensor([ 1.2481,  0.7891,  0.6946, -0.5056, -0.3829,  0.1210,  2.7367, -0.1867,
        -0.6069,  0.1659,  0.1566,  0.8303, -1.3156, -0.5701,  0.9452,  0.3261],
       grad_fn=<SliceBackward0>)
loss = 1.9711116552352905
tensor([ 1.2385,  0.7978,  0.7008, -0.5156, -0.3929,  0.1111,  2.7328, -0.1767,
        -0.5970,  0.1738,  0.1468,  0.8403, -1.3256, -0.5660,  0.9352,  0.3211],
       grad_fn=<SliceBackward0>)
loss = 1.92123544216156
tensor([ 1.2287,  0.8058,  0.7029, -0.5256, -0.4030,  0.1014,  2.7323, -0.1667,
        -0.5872,  0.1810,  0.1369,  0.8504, -1.3357, -0.5641,  0.9252,  0.3187],
       grad_fn=<SliceBackward0>)
loss = 1.8763824701309204
tensor([ 1.2188,  0.8134,  0.7013, -0.5356, -0.4130,  0.0917,  2.7339, -0.1566,
        -0.5775,  0.1881,  0.1270,  0.8604